# Configurable streaming comparison
Initially up to five separate customer streams; if the archive is unavailable, five independent stationary synthetic streams demonstrate the machinery only. These synthetic generator seeds are replicates of the process, not neural initialization seeds. DNH-inspired v1, its fixed counterpart and separately disabled growth/pruning/frequency mechanisms are run in the appended bounded study. Faithful DNH remains unimplemented; `docs/DNH_INSPIRED.md` explains the distinction.

Each per-series directory retains prediction CSV, failure events, elapsed time, sampled RSS, full configuration and resumable checkpoint. Results across series are macro averaged using development-scaled MASE so high consumption does not dominate. Report per-series values first. Adjacent hours are not independent replicates; no IID confidence intervals are produced.

In [ ]:
from pathlib import Path
import os, sys, json, time
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'research').is_dir())
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
os.environ.setdefault('OMP_NUM_THREADS','1')
os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from research.experiment import config, hardware, run_stream, metrics
from research.data import acquire, parse_tsf, audit, select
from research.synthetic import processes, change_analysis
from research.inspired_experiment import run_inspired, inspired_config
from research.inspired_study import compare_streams, controlled
from research.validation import source, electricity_source
MODE = os.environ.get('EXPERIMENT_MODE','quick')
cfg = config(MODE)
OUT = Path(os.environ.get('DNH_RESULTS_DIR',ROOT/'results'/MODE))
OUT.mkdir(parents=True, exist_ok=True)
print(hardware()); print(cfg)
def stream_provenance(sid, values, generator_seed=17):
    if series:
        return electricity_source({'id':sid,'values':values},verified)
    return source('synthetic',generator_seed=generator_seed,process='stationary')
archive = sorted((ROOT/'data/raw').glob('*.zip'))
series = []
if archive:
    verified = acquire(ROOT/'data/raw', manual=True)
    metadata, series = parse_tsf(verified)
    print(metadata)
    data_audit = audit(series)
    display(data_audit)
    data_audit.to_csv(OUT/'electricity_audit.csv', index=False)
    print('Actual series count:', len(series), 'lengths:', data_audit.length.value_counts().to_dict())
    print('Claim checks (not assumed):', {'321_series':len(series)==321,
          'all_start_2012':data_audit.start.str.startswith('2012').all(),
          'all_end_2014':data_audit.end.str.startswith('2014').all()})
else:
    print('ELECTRICITY BLOCKED: no source-verified archive. No electricity results generated.')


In [ ]:
chosen=select(series,cfg) if series else []
streams=[(s['id'],s['values'],s['index']) for s in chosen]
if not streams:
    streams=[(f'synthetic_{seed}',processes(cfg,seed)['stationary']['values'],None) for seed in range(100,100+cfg['series'])]
summary=[]
for sid,y,index in streams:
    pred,report=run_stream(y,cfg,OUT/'comparison'/sid,timestamps=index,provenance=stream_provenance(sid,y,int(sid.split('_')[-1]) if sid.startswith('synthetic_') else 17))
    m=metrics(pred); m['series']=sid; summary.append(m)
    for model,g in pred.groupby('model'):
        plt.plot(g.t,g.scaled_abs_error.rolling(12,min_periods=1).mean(),alpha=.5,label=f'{sid}:{model}')
summary=pd.concat(summary,ignore_index=True)
summary.to_csv(OUT/'individual_series_metrics.csv',index=False)
display(summary)
aggregate=summary.groupby('model').agg(macro_MASE=('MASE','mean'),series_count=('series','count'))
display(aggregate); aggregate.to_csv(OUT/'macro_metrics.csv')
plt.ylabel('trailing scaled absolute error'); plt.title('Independent streams: '+('electricity' if chosen else 'SYNTHETIC diagnostics'))
plt.savefig(OUT/'comparison_rolling.png',bbox_inches='tight'); plt.show()

DNH-inspired experiments below use three model seeds, separate from synthetic generator seeds. Average seeds within each customer before macro aggregation. Baselines are deterministic and are not repeated under model seeds. Optional SARIMA remains unrun. Tiny quick horizons do not support population conclusions.

## DNH-inspired comparison and bounded ablations
The same separately selected streams are used. Adaptive and fixed models share each seed and initial state; model-seed spread and customer spread are saved separately. The fixed model disables structure and frequency changes, retaining seeded nested memories. Ablations disable growth, pruning or frequency adaptation separately, on one customer and one model seed only. Counts of accepted events and rejected proposals prevent attributing benefits to a mechanism that never activated. These are diagnostic ablations, not a tuned study.

In [ ]:
individual,macro,costs=compare_streams(streams,cfg,OUT/'inspired_comparison',provenance_by_series={sid:stream_provenance(sid,y,int(sid.split('_')[-1]) if sid.startswith('synthetic_') else 17) for sid,y,_ in streams})
display(individual);display(macro);display(costs)
ablations=compare_streams(streams[:inspired_config()['ablation_series']],cfg,OUT/'inspired_ablations',
                         seeds=inspired_config()['ablation_seeds'],
                         variants=['adaptive','fixed','no_growth','no_pruning','no_frequency'],provenance_by_series={sid:stream_provenance(sid,y,int(sid.split('_')[-1]) if sid.startswith('synthetic_') else 17) for sid,y,_ in streams})
display(ablations[1]);display(ablations[2])
# Each plot retains customer and seed identity; adjacent timestamps are not replicates.
for sid,_,_ in streams:
    fig,ax=plt.subplots(figsize=(10,3))
    for model_seed in inspired_config()['seeds']:
        for variant in ['adaptive','fixed']:
            p=pd.read_csv(OUT/'inspired_comparison'/sid/str(model_seed)/variant/'predictions.csv')
            ax.plot(p.t,p.scaled_abs_error.rolling(12,min_periods=1).mean(),label=f'{variant} seed {model_seed}',alpha=.7)
    ax.set_title(sid+' DNH-inspired rolling errors');ax.set_ylabel('scaled absolute error');ax.legend(fontsize=7)
    fig.savefig(OUT/f'inspired_rolling_{sid}.png',bbox_inches='tight');plt.show()